In [2]:
import sqlite3
import pandas as pd
from pathlib import Path

# Locate the database
possible_paths = [
    Path("../backend/database/cinematch.db"),
    Path("../database/cinematch.db"),
    Path("backend/database/cinematch.db")
]

DB_PATH = next((p.resolve() for p in possible_paths if p.exists()), None)

if DB_PATH is None:
    raise FileNotFoundError("Could not locate cinematch.db")

print("Database found:", DB_PATH)

Database found: /Users/pragya.sharma/Documents/CPSC481_Final/backend/database/cinematch.db


In [4]:
# collecting each movie's details along with its tags in a single column

conn = sqlite3.connect(DB_PATH)

query = """
SELECT
    m.movie_id,
    m.title,
    m.year,
    m.genres,
    COALESCE(GROUP_CONCAT(t.tag, ' '), '') AS tags
FROM movies m
LEFT JOIN tags t ON m.movie_id = t.movie_id
GROUP BY m.movie_id
ORDER BY m.movie_id;
"""

movies_df = pd.read_sql_query(query, conn)

conn.close()

movies_df.head()

,movie_id,title,year,genres,tags
0,1,Toy Story (1995),1995.0,Adventure|Animation|Children|Comedy|Fantasy,fun pixar pixar
1,2,Jumanji (1995),1995.0,Adventure|Children|Fantasy,fantasy game magic board game robin williams
2,3,Grumpier Old Men (1995),1995.0,Comedy|Romance,moldy old
3,4,Waiting to Exhale (1995),1995.0,Comedy|Drama|Romance,
4,5,Father of the Bride Part II (1995),1995.0,Comedy,pregnancy remake


In [ ]:
# running the extracted data to check if the data is correct and the number of rows and columns are correct

print("Number of movies:", len(movies_df))
print("Columns:", movies_df.columns.tolist())

movies_df.head(10)

Number of movies: 9742
Columns: ['movie_id', 'title', 'year', 'genres', 'tags']


,movie_id,title,year,genres,tags
0,1,Toy Story (1995),1995.0,Adventure|Animation|Children|Comedy|Fantasy,fun pixar pixar
1,2,Jumanji (1995),1995.0,Adventure|Children|Fantasy,fantasy game magic board game robin williams
2,3,Grumpier Old Men (1995),1995.0,Comedy|Romance,moldy old
3,4,Waiting to Exhale (1995),1995.0,Comedy|Drama|Romance,
4,5,Father of the Bride Part II (1995),1995.0,Comedy,pregnancy remake
5,6,Heat (1995),1995.0,Action|Crime|Thriller,
6,7,Sabrina (1995),1995.0,Comedy|Romance,remake
7,8,Tom and Huck (1995),1995.0,Adventure|Children,
8,9,Sudden Death (1995),1995.0,Action,
9,10,GoldenEye (1995),1995.0,Action|Adventure|Thriller,


In [6]:
# Handle missing values
movies_df["genres"] = movies_df["genres"].fillna("")
movies_df["tags"] = movies_df["tags"].fillna("")

# Convert genres from "|" separated format to spaces
movies_df["genres_clean"] = (
    movies_df["genres"]
    .str.replace("|", " ", regex=False)
    .str.lower()
    .str.strip()
)

# Clean tags
movies_df["tags_clean"] = (
    movies_df["tags"]
    .str.lower()
    .str.strip()
)

movies_df[["title", "genres_clean", "tags_clean"]].head(10)

,title,genres_clean,tags_clean
0,Toy Story (1995),adventure animation children comedy fantasy,fun pixar pixar
1,Jumanji (1995),adventure children fantasy,fantasy game magic board game robin williams
2,Grumpier Old Men (1995),comedy romance,moldy old
3,Waiting to Exhale (1995),comedy drama romance,
4,Father of the Bride Part II (1995),comedy,pregnancy remake
5,Heat (1995),action crime thriller,
6,Sabrina (1995),comedy romance,remake
7,Tom and Huck (1995),adventure children,
8,Sudden Death (1995),action,
9,GoldenEye (1995),action adventure thriller,


In [ ]:
# combining the genres and tags into a single column for each movie to create a movie profile

movies_df["movie_profile"] = (
    movies_df["genres_clean"] + " " + movies_df["tags_clean"]
).str.strip()

movies_df[["title", "movie_profile"]].head(10)

,title,movie_profile
0,Toy Story (1995),adventure animation children comedy fantasy fu...
1,Jumanji (1995),adventure children fantasy fantasy game magic ...
2,Grumpier Old Men (1995),comedy romance moldy old
3,Waiting to Exhale (1995),comedy drama romance
4,Father of the Bride Part II (1995),comedy pregnancy remake
5,Heat (1995),action crime thriller
6,Sabrina (1995),comedy romance remake
7,Tom and Huck (1995),adventure children
8,Sudden Death (1995),action
9,GoldenEye (1995),action adventure thriller


In [8]:
# Check for missing profiles
print("Missing profiles:", movies_df["movie_profile"].isna().sum())

# Check for empty profiles
empty_profiles = movies_df["movie_profile"].str.strip().eq("").sum()
print("Empty profiles:", empty_profiles)

# Inspect a random sample
movies_df[["title", "movie_profile"]].sample(10, random_state=42)

Missing profiles: 0
Empty profiles: 0


,title,movie_profile
6213,"Notorious Bettie Page, The (2005)",drama
1056,Star Trek V: The Final Frontier (1989),action sci-fi
4891,Starsky & Hutch (2004),action comedy crime thriller
346,"Colonel Chabert, Le (1994)",drama romance war
5353,"Country Girl, The (1954)",drama oscar (best actress)
7829,"Civil War, The (1990)",documentary war
8655,John Mulaney: New In Town (2012),comedy
9056,Saving Christmas (2014),children comedy
1111,Kolya (Kolja) (1996),comedy drama in netflix queue
7278,Alice in Wonderland (2010),adventure fantasy imax


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize the TF-IDF vectorizer
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

# Transform movie profiles into numerical vectors
tfidf_matrix = tfidf.fit_transform(movies_df["movie_profile"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (9742, 5484)


In [10]:
print("Number of movies:", tfidf_matrix.shape[0])
print("Number of unique features:", tfidf_matrix.shape[1])

Number of movies: 9742
Number of unique features: 5484


In [12]:
feature_names = tfidf.get_feature_names_out()

print("First 50 features:")
print(feature_names[:100])

First 50 features:
['06' '06 oscar' '1900s' '1920s' '1920s gangsters' '1950s'
 '1950s adolescence' '1960s' '1960s jim' '1970s' '1970s movies'
 '1970s sofia' '1980s' '1980s high' '1990s' '1990s achronological' '2001'
 '2001 like' '250' '250 innovative' '250 james' '250 jean' '250 jeff'
 '250 masterpiece' '250 mental' '250 mindfuck' '250 mount' '250 norman'
 '250 revenge' '2d' '2d animation' '70mm' '80' 'aardman' 'abortion'
 'absorbing' 'absorbing acting' 'abstract' 'abstract dark' 'abuse'
 'abuse kidnapping' 'abuse narrated' 'abuse psychology' 'academy'
 'academy award' 'accident' 'achronological' 'achronological action'
 'acting' 'acting atmospheric' 'acting bad' 'acting great' 'acting guns'
 'acting helena' 'acting interesting' 'action' 'action action'
 'action adventure' 'action aliens' 'action alternate' 'action animation'
 'action artificial' 'action assassin' 'action atmospheric'
 'action batman' 'action children' 'action choreography'
 'action christoph' 'action classic' 'action 

In [13]:
import numpy as np

movie_title = "Toy Story (1995)"

movie_index = movies_df.index[
    movies_df["title"] == movie_title
][0]

movie_vector = tfidf_matrix[movie_index]

# Extract nonzero features and their weights
feature_indices = movie_vector.nonzero()[1]
feature_weights = movie_vector.data

movie_features = pd.DataFrame({
    "feature": feature_names[feature_indices],
    "tfidf_weight": feature_weights
}).sort_values("tfidf_weight", ascending=False)

movie_features.head(15)

,feature,tfidf_weight
6,pixar,0.621950
11,fantasy fun,0.344205
12,fun pixar,0.344205
13,pixar pixar,0.329500
5,fun,0.298772
10,comedy fantasy,0.171192
7,adventure animation,0.169247
8,animation children,0.162129
9,children comedy,0.155577
1,animation,0.136634
